## Modelado Gamma Gamma

In [6]:
# ============================================
# LIBRERÍAS REQUERIDAS
# ============================================
import pandas as pd
import numpy as np
from lifetimes import GammaGammaFitter  # Modelo Gamma-Gamma para valor monetario
import warnings
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr, spearmanr

warnings.filterwarnings('ignore')

print("\n" + "=" * 100)
print("ANÁLISIS CLV MEDIANTE MODELADO GAMMA-GAMMA")
print("=" * 100)

# ============================================
# 1. CARGA Y LIMPIEZA DE DATOS
# ============================================
# Cargar dataset consolidado a nivel cliente con métricas RFM y CLV
customer_df = pd.read_csv("./data/customer_clv.csv")

# Eliminar outliers extremos para estabilidad del modelo
# Mantener percentil 5% - 95% para análisis robustos
customer_df_clean = customer_df.copy()
customer_df_clean = customer_df_clean[customer_df_clean["CLV"] <= customer_df_clean["CLV"].quantile(0.95)]
customer_df_clean = customer_df_clean[customer_df_clean["CLV"] >= customer_df_clean["CLV"].quantile(0.05)]

print(f"\nClientes tras limpieza de outliers (5%-95%): {len(customer_df_clean):,}")
print(f"Rango CLV analizado: {customer_df_clean['CLV'].min():.2f} - {customer_df_clean['CLV'].max():.2f}")

# ============================================
# 2. DEFINICIÓN DINÁMICA DE SEGMENTOS CLV
# ============================================
# Crear segmentos automáticos basados en percentiles de CLV
percentiles = [0, 5, 15, 30, 50, 65, 80, 90, 95, 99, 100]

# Calcular valores de corte en percentiles
cuts = np.percentile(customer_df_clean["CLV"], percentiles)

# Crear nombres descriptivos para segmentos dinámicos
labels = [
    'Bajo_extremo',
    'Bajo_medio',
    'Medio_bajo',
    'Medio',
    'Medio_alto',
    'Alto_medio',
    'Alto',
    'Alto_superior',
    'Premium',
    'Top'
]

# Construcción del diccionario de segmentos granulares
brackets = {
    label: (cuts[i], cuts[i+1])
    for i, label in enumerate(labels)
}

# ============================================
# 3. SEGMENTOS AMPLIOS PARA ANÁLISIS
# ============================================
# Agrupar segmentos granulares en categorías más amplias
brackets_amplios = {
    'Todos_clientes': (cuts[0], cuts[-1]),  # Población completa
    'Bajo_valor': (cuts[0], cuts[3]),       # 0-30%
    'Alto_valor': (cuts[3], cuts[-1]),      # 30%+
    'Core_business': (cuts[2], cuts[7]),    # 15%-90% (cliente principal)
    'High_value': (cuts[7], cuts[-1]),      # Top 10%
}

# Mostrar segmentos configurados
print(f"\n=== SEGMENTOS GRANULARES DEFINIDOS ===")
for k, v in brackets.items():
    print(f"  {k}: {v[0]:.2f} - {v[1]:.2f}")


ANÁLISIS CLV MEDIANTE MODELADO GAMMA-GAMMA

Clientes tras limpieza de outliers (5%-95%): 627,307
Rango CLV analizado: 30.76 - 2467.82

=== SEGMENTOS GRANULARES DEFINIDOS ===
  Bajo_extremo: 30.76 - 42.99
  Bajo_medio: 42.99 - 81.07
  Medio_bajo: 81.07 - 143.35
  Medio: 143.35 - 254.84
  Medio_alto: 254.84 - 403.96
  Alto_medio: 403.96 - 727.75
  Alto: 727.75 - 1119.29
  Alto_superior: 1119.29 - 1550.97
  Premium: 1550.97 - 2177.24
  Top: 2177.24 - 2467.82


In [7]:
# ============================================
# 4. ANÁLISIS DE VALIDEZ POR SEGMENTO
# ============================================
# Combinar segmentos granulares y amplios para análisis completo
todos_brackets = {**brackets, **brackets_amplios}

print("\n" + "=" * 100)
print("SEGMENTACIÓN DE CLIENTES Y VALIDEZ PARA MODELO GAMMA-GAMMA")
print("=" * 100)

# Preparar análisis por segmento
table_data = []
for name, (lower, upper) in todos_brackets.items():
    # Filtrar clientes en rango de CLV
    mask = (customer_df_clean["CLV"] >= lower) & (customer_df_clean["CLV"] <= upper)
    population = mask.sum()
    pct = (population / len(customer_df_clean)) * 100
    
    # Contar clientes válidos para Gamma-Gamma (con frequency > 0 y monetary > 0)
    valid_mask = mask & (customer_df_clean["frequency"] > 0) & (customer_df_clean["monetary"] > 0)
    valid_pop = valid_mask.sum()
    valid_pct = (valid_pop / population * 100) if population > 0 else 0
    
    # Evaluar suficiencia de volumen para entrenamiento
    if valid_pop >= 100:
        estado = "Suficiente"
    elif valid_pop >= 30:
        estado = "Limitado"
    else:
        estado = "Insuficiente"
    
    table_data.append([name, f"{lower:.0f}-{upper:.0f}", f"{population:,}", f"{pct:.1f}%", 
                       f"{valid_pop:,}", f"{valid_pct:.1f}%", estado])

# Mostrar tabla de segmentación
print(f"\n{'Segmento':<20} {'Rango CLV':<15} {'Clientes':<12} {'%':<10} {'Válidos':<12} {'% Válidos':<11} {'Estado':<15}")
print("-" * 100)
for row in table_data:
    print(f"{row[0]:<20} {row[1]:<15} {row[2]:<12} {row[3]:<10} {row[4]:<12} {row[5]:<11} {row[6]:<15}")


SEGMENTACIÓN DE CLIENTES Y VALIDEZ PARA MODELO GAMMA-GAMMA

Segmento             Rango CLV       Clientes     %          Válidos      % Válidos   Estado         
----------------------------------------------------------------------------------------------------
Bajo_extremo         31-43           31,387       5.0%       31,387       100.0%      Suficiente     
Bajo_medio           43-81           64,128       10.2%      64,128       100.0%      Suficiente     
Medio_bajo           81-143          94,136       15.0%      94,136       100.0%      Suficiente     
Medio                143-255         125,485      20.0%      125,485      100.0%      Suficiente     
Medio_alto           255-404         94,105       15.0%      94,105       100.0%      Suficiente     
Alto_medio           404-728         94,101       15.0%      94,101       100.0%      Suficiente     
Alto                 728-1119        62,732       10.0%      62,732       100.0%      Suficiente     
Alto_superior        1

In [8]:
# ============================================
# 5. MODELADO GAMMA-GAMMA POR SEGMENTO
# ============================================
# Gamma-Gamma: Modelar componente monetario del CLV
# Supuesto: Independencia entre frecuencia de compra y valor monetario por transacción

print("\n" + "=" * 100)
print("ENTRENAMIENTO DEL MODELO GAMMA-GAMMA POR SEGMENTO CLV")
print("=" * 100)

results = {}
tabla_resultados = []
tabla_errores = []

print("\n" + "=" * 80)
print("MODELADO GAMMA-GAMMA POR SEGMENTO DE CLV")
print("=" * 80)

for bracket_name, (lower, upper) in todos_brackets.items():
    print(f"\n{'─' * 80}")
    print(f"SEGMENTO: {bracket_name} | Rango CLV: [{lower:.0f} - {upper:.0f}]")
    print(f"{'─' * 80}")
    
    # Filtrar clientes en rango CLV
    bracket_df = customer_df_clean[
        (customer_df_clean["CLV"] >= lower) & 
        (customer_df_clean["CLV"] <= upper)
    ].copy()
    
    # Seleccionar clientes válidos (frequency > 0, monetary > 0)
    valid_customers = bracket_df[
        (bracket_df["frequency"] > 0) & 
        (bracket_df["monetary"] > 0)
    ].copy()
    
    print(f"Clientes totales: {len(bracket_df):,}")
    print(f"Clientes válidos: {len(valid_customers):,}")
    
    # Validación de volumen mínimo
    if len(valid_customers) < 30:
        print(f"Estado: DESCARTADO (volumen insuficiente para entrenar modelo)")
        results[bracket_name] = None
        tabla_resultados.append({
            'Segmento': bracket_name,
            'Rango_CLV': f"[{lower:.0f} - {upper:.0f}]",
            'Clientes_Validos': len(valid_customers),
            'Estado': 'Descartado',
            'Correlación': None,
            'Monetary_Esperado': None,
            'Parámetros': None
        })
        continue
    
    gg_df = valid_customers[["frequency", "monetary"]].copy()

    # Mostrar resumen estadístico del segmento
    print(f"\n  RESUMEN ESTADÍSTICO DEL SEGMENTO:")
    print(f"    • CLV promedio: {bracket_df['CLV'].mean():.2f}")
    print(f"    • CLV mediana: {bracket_df['CLV'].median():.2f}")
    print(f"    • Frecuencia promedio: {gg_df['frequency'].mean():.2f} compras")
    print(f"    • Monetary promedio: {gg_df['monetary'].mean():.2f}")
    
    

    # VALIDACIÓN CLAVE: Independencia entre frequency y monetary
    # Gamma-Gamma requiere correlación baja (|r| < 0.3)
    corr = gg_df["frequency"].corr(gg_df["monetary"])
    
    print(f"\n  VALIDACIÓN DE INDEPENDENCIA (Gamma-Gamma requirement):")
    print(f"    • Correlación (frequency vs monetary): {corr:.4f}")
    
    if abs(corr) < 0.3:
        print(f"    Independencia confirmada -> Entrenar modelo")
        
        try:
            # Intentar múltiples penalizers para encontrar mejor fit
            penalizers = [0.0, 0.001, 0.01, 0.1, 1.0]
            model_trained = False
            
            for penalizer in penalizers:
                try:
                    # Ajustar modelo Gamma-Gamma
                    ggf = GammaGammaFitter(penalizer_coef=penalizer)
                    ggf.fit(
                        gg_df["frequency"], 
                        gg_df["monetary"],
                        verbose=False
                    )
                    
                    # Validar parámetros (deben ser finitos y positivos)
                    if all(np.isfinite(ggf.params_)) and all(ggf.params_ > 0):
                        # Calcular valor monetario esperado
                        conditional_exp = ggf.conditional_expected_average_profit(
                            gg_df["frequency"], 
                            gg_df["monetary"]
                        )
                        
                        results[bracket_name] = {
                            'model': ggf,
                            'correlation': corr,
                            'n_customers': len(valid_customers),
                            'n_clean': len(gg_df),
                            'avg_conditional_exp': conditional_exp.mean(),
                            'median_conditional_exp': conditional_exp.median(),
                            'std_conditional_exp': conditional_exp.std(),
                            'params': {
                                'p': ggf.params_[0],
                                'q': ggf.params_[1],
                                'v': ggf.params_[2]
                            },
                            'penalizer': penalizer,
                            'clv_range': (lower, upper),
                            'avg_clv': bracket_df['CLV'].mean(),
                            'median_clv': bracket_df['CLV'].median()
                        }
                        
                        # Mostrar parámetros del modelo
                        print(f"\n  PARÁMETROS DEL MODELO (penalizer={penalizer}):")
                        print(f"    • p (shape): {ggf.params_[0]:.4f}")
                        print(f"    • q (rate): {ggf.params_[1]:.4f}")
                        print(f"    • v (scale): {ggf.params_[2]:.4f}")
                        print(f"\n  VALOR MONETARIO ESPERADO:")
                        print(f"    • Media: {conditional_exp.mean():.2f}")
                        print(f"    • Mediana: {conditional_exp.median():.2f}")
                        print(f"    • Desv. Estándar: {conditional_exp.std():.2f}")
                        
                        tabla_resultados.append({
                            'Segmento': bracket_name,
                            'Rango_CLV': f"[{lower:.0f} - {upper:.0f}]",
                            'Clientes_Validos': len(valid_customers),
                            'Estado': 'Entrenado',
                            'Correlación': f"{corr:.4f}",
                            'Monetary_Esperado': f"{conditional_exp.mean():.2f}",
                            'Parámetros': f"p={ggf.params_[0]:.2f}, q={ggf.params_[1]:.2f}, v={ggf.params_[2]:.2f}"
                        })

                        # Predicción monetary esperado
                        y_true = gg_df["monetary"]
                        y_pred = conditional_exp.copy()


                        mae = mean_absolute_error(y_true, y_pred)
                        mape = mean_absolute_error(y_true, y_pred) / y_true.mean() * 100
                        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
                        r2 = r2_score(y_true, y_pred)
                        corr, _ = pearsonr(y_true, y_pred)
                        spearman = spearmanr(y_true, y_pred).correlation

                        print("\n=== MÉTRICAS DE ERROR DEL GAMMA-GAMMA ===\n")

                        print(f"MAE: {mae:.2f}")
                        print(f"MAPE: {mape:.2f}%")
                        print(f"RMSE: {rmse:.2f}")
                        print(f"R2: {r2:.3f}")
                        print(f"Pearson corr: {corr:.3f}")
                        print(f"Spearman corr: {spearman:.3f}")

                        tabla_errores.append({
                            "Segmento": bracket_name,
                            "Rango_CLV": f"[{lower:.0f} - {upper:.0f}]",
                            "Clientes": len(gg_df),

                            "MAE": round(mae, 2),
                            "MAPE": round(mape, 2),
                            "RMSE": round(rmse, 2),
                            "R2": round(r2, 3),

                            "Pearson": round(corr, 3),
                            "Spearman": round(spearman, 3),

                            "Monetary_Mean": round(y_true.mean(), 2),
                            "Predicted_Mean": round(y_pred.mean(), 2),

                            "Penalizer": penalizer
                        })


                        model_trained = True
                        break
                except:
                    continue
            
            if not model_trained:
                print(f"No se pudo ajustar un modelo estable con ningún penalizer")
                results[bracket_name] = None
                tabla_resultados.append({
                    'Segmento': bracket_name,
                    'Rango_CLV': f"[{lower:.0f} - {upper:.0f}]",
                    'Clientes_Validos': len(valid_customers),
                    'Estado': 'Error',
                    'Correlación': f"{corr:.4f}",
                    'Monetary_Esperado': None,
                    'Parámetros': None
                })
                
        except Exception as e:
            print(f"   Excepción en entrenamiento: {str(e)[:60]}")
            results[bracket_name] = None
            tabla_resultados.append({
                'Segmento': bracket_name,
                'Rango_CLV': f"[{lower:.0f} - {upper:.0f}]",
                'Clientes_Validos': len(valid_customers),
                'Estado': 'Excepción',
                'Correlación': f"{corr:.4f}",
                'Monetary_Esperado': None,
                'Parámetros': None
            })
    else:
        print(f"     No independencia (|r| ≥ 0.3) -> Modelo no aplicable para este segmento")
        results[bracket_name] = None
        tabla_resultados.append({
            'Segmento': bracket_name,
            'Rango_CLV': f"[{lower:.0f} - {upper:.0f}]",
            'Clientes_Validos': len(valid_customers),
            'Estado': 'No independencia',
            'Correlación': f"{corr:.4f}",
            'Monetary_Esperado': None,
            'Parámetros': None
        })

# ============================================
# 6. RESUMEN FINAL DE MODELADO
# ============================================
print(f"\n{'=' * 100}")
print("RESUMEN FINAL: RESULTADOS DEL MODELADO GAMMA-GAMMA")
print(f"{'=' * 100}")

tabla_final = pd.DataFrame(tabla_resultados)
print(f"\n{tabla_final.to_string(index=False)}")

# Mostrar solo segmentos exitosos
exitosos = [r for r in tabla_resultados if "Entrenado" in r['Estado']]
if exitosos:
    print(f"\n{'=' * 100}")
    print("SEGMENTOS CON MODELOS EXITOSOS")
    print(f"{'=' * 100}")
    tabla_exitosos = pd.DataFrame(exitosos)
    print(f"\n{tabla_exitosos.to_string(index=False)}")


ENTRENAMIENTO DEL MODELO GAMMA-GAMMA POR SEGMENTO CLV

MODELADO GAMMA-GAMMA POR SEGMENTO DE CLV

────────────────────────────────────────────────────────────────────────────────
SEGMENTO: Bajo_extremo | Rango CLV: [31 - 43]
────────────────────────────────────────────────────────────────────────────────
Clientes totales: 31,387
Clientes válidos: 31,387

  RESUMEN ESTADÍSTICO DEL SEGMENTO:
    • CLV promedio: 37.26
    • CLV mediana: 37.84
    • Frecuencia promedio: 1.07 compras
    • Monetary promedio: 37.26

  VALIDACIÓN DE INDEPENDENCIA (Gamma-Gamma requirement):
    • Correlación (frequency vs monetary): -0.0234
    Independencia confirmada -> Entrenar modelo

  PARÁMETROS DEL MODELO (penalizer=0.0):
    • p (shape): 171.8808
    • q (rate): 225.8517
    • v (scale): 48.7364

  VALOR MONETARIO ESPERADO:
    • Media: 37.25
    • Mediana: 37.51
    • Desv. Estándar: 1.64

=== MÉTRICAS DE ERROR DEL GAMMA-GAMMA ===

MAE: 1.78
MAPE: 4.78%
RMSE: 2.05
R2: 0.688
Pearson corr: 0.995
Spearma

In [9]:
# ============================================
# TABLA FINAL DE ERRORES
# ============================================

print(f"\n{'=' * 100}")
print("RESUMEN FINAL: MÉTRICAS DE ERROR POR SEGMENTO")
print(f"{'=' * 100}")

tabla_errores_df = pd.DataFrame(tabla_errores)

print("\n")
print(tabla_errores_df.to_string(index=False))


RESUMEN FINAL: MÉTRICAS DE ERROR POR SEGMENTO


     Segmento     Rango_CLV  Clientes   MAE  MAPE  RMSE    R2  Pearson  Spearman  Monetary_Mean  Predicted_Mean  Penalizer
 Bajo_extremo     [31 - 43]     31387  1.78  4.78  2.05 0.688    0.995     0.997          37.26           37.25      0.000
   Bajo_medio     [43 - 81]     64128  1.25  2.10  1.47 0.982    1.000     1.000          59.65           59.72      0.000
   Medio_bajo    [81 - 143]     94136  5.35  4.68  6.10 0.888    0.995     0.996         114.32          114.17      0.000
        Medio   [143 - 255]    125485  5.69  2.90  6.03 0.967    0.999     0.998         196.24          201.94      0.001
   Medio_alto   [255 - 404]     94105  8.67  2.70  9.33 0.956    0.997     0.996         320.88          329.54      0.001
   Alto_medio   [404 - 728]     94101 13.56  2.51 15.19 0.972    0.997     0.997         539.97          553.53      0.001
         Alto  [728 - 1119]     62732 25.17  2.78 28.94 0.931    0.992     0.991         9

In [10]:
# ============================================
# 7. TABLA CONSOLIDADA DE RESULTADOS
# ============================================
# Mostrar solo segmentos con modelo entrenado exitosamente

print(f"\n{'=' * 100}")
print("SEGMENTOS CON MODELOS MONETARIOS VÁLIDOS")
print(f"{'=' * 100}")

# Filtrar modelos exitosos
successful = [(name, res) for name, res in results.items() if res is not None]

if successful:
    # Construir tabla con métricas detalladas
    table_data = []
    for name, res in successful:
        table_data.append({
            'Segmento': name,
            'Rango CLV': f"[{res['clv_range'][0]:.0f} - {res['clv_range'][1]:.0f}]",
            'Clientes': f"{res['n_customers']:,}",
            'Correlación': f"{res['correlation']:.4f}",
            'CLV Promedio': f"{res['avg_clv']:.2f}",
            'CLV Mediana': f"{res['median_clv']:.2f}",
            'Monetary Esperado': f"{res['avg_conditional_exp']:.2f}",
            'Monetary Mediana': f"{res['median_conditional_exp']:.2f}",
            'Parámetros (p,q,v)': f"({res['params']['p']:.2f}, {res['params']['q']:.2f}, {res['params']['v']:.2f})"
        })
    
    df_table = pd.DataFrame(table_data)
    
    # Configurar pandas para mostrar tabla completa
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    pd.set_option('display.max_colwidth', 30)
    
    print(f"\n{df_table.to_string(index=False)}")
    
    # Resumen global
    print(f"\n{'=' * 100}")
    print("RESUMEN GLOBAL")
    print('='*100)
    
    summary_data = [
        ['Segmentos analizados', f"{len(todos_brackets)}"],
        ['Segmentos válidos', f"{len(successful)}"],
        ['Ratio de aplicabilidad', f"{len(successful)/len(todos_brackets)*100:.1f}%"]
    ]
    
    summary_df = pd.DataFrame(summary_data, columns=['Métrica', 'Valor'])
    print(summary_df.to_string(index=False))
    
    # Exportar resultados
    results_df = pd.DataFrame([{
        'bracket': name,
        'clv_lower': res['clv_range'][0],
        'clv_upper': res['clv_range'][1],
        'n_customers': res['n_customers'],
        'correlation': res['correlation'],
        'avg_clv': res['avg_clv'],
        'median_clv': res['median_clv'],
        'avg_monetary_pred': res['avg_conditional_exp'],
        'median_monetary_pred': res['median_conditional_exp'],
        'p_param': res['params']['p'],
        'q_param': res['params']['q'],
        'v_param': res['params']['v'],
        'penalizer': res['penalizer']
    } for name, res in successful])
    
    
    print("\nVista previa de datos")
    print(results_df.head().to_string(index=False))

else:
    print("\nNo se han identificado segmentos adecuados para Gamma-Gamma")


SEGMENTOS CON MODELOS MONETARIOS VÁLIDOS

     Segmento     Rango CLV Clientes Correlación CLV Promedio CLV Mediana Monetary Esperado Monetary Mediana      Parámetros (p,q,v)
 Bajo_extremo     [31 - 43]   31,387     -0.0234        37.26       37.84             37.25            37.51 (171.88, 225.85, 48.74)
   Bajo_medio     [43 - 81]   64,128      0.1594        59.65       59.18             59.72            59.26   (194.43, 33.34, 9.94)
   Medio_bajo    [81 - 143]   94,136     -0.0598       114.32      117.12            114.17           116.28   (93.76, 54.65, 65.34)
        Medio   [143 - 255]  125,485      0.0453       196.24      191.77            201.94           197.29    (15.06, 1.58, 14.45)
   Medio_alto   [255 - 404]   94,105      0.1043       320.88      315.03            329.54           322.40    (14.92, 1.13, 14.54)
   Alto_medio   [404 - 728]   94,101      0.1329       539.97      525.84            553.53           536.38    (14.47, 0.80, 14.24)
         Alto  [728 - 1119